In [3]:
import torch
import torch.nn as nn

In [4]:
class DenseHighLevel(nn.Module):

  def __init__(self, in_features, out_features):
    super().__init__()

    self.linear = nn.Linear(in_features, out_features)
    self.relu = nn.ReLU()

  def forward(self, x):
    x = self.linear(x)
    x = self.relu(x)

    return x


model = DenseHighLevel(in_features=10, out_features=5)
print(model)



DenseHighLevel(
  (linear): Linear(in_features=10, out_features=5, bias=True)
  (relu): ReLU()
)


In [5]:
for i in model.parameters():
  print(i)

Parameter containing:
tensor([[-0.2087,  0.3016,  0.3139, -0.1067, -0.2330,  0.2539,  0.1664, -0.2853,
          0.1923,  0.3134],
        [-0.2228, -0.1225,  0.1593,  0.2812,  0.2109,  0.2426, -0.2664,  0.2499,
         -0.2784, -0.0552],
        [-0.0351, -0.1589, -0.3040, -0.3104,  0.1127,  0.2901, -0.1090, -0.2314,
         -0.1123,  0.1425],
        [ 0.0743, -0.0917,  0.1346, -0.2282,  0.2300,  0.1769, -0.0798,  0.1145,
         -0.1927, -0.0708],
        [ 0.1446, -0.2074, -0.2948,  0.2686,  0.1803, -0.0242, -0.0068,  0.1226,
          0.0110,  0.1021]], requires_grad=True)
Parameter containing:
tensor([ 0.1451, -0.1441, -0.1375, -0.2888,  0.2508], requires_grad=True)


In [6]:
import torch.nn.functional as F


class DenseLowLevel(nn.Module):

  def __init__(self, in_features, out_features):
    super().__init__()

    self.weights = nn.Parameter(torch.randn(out_features, in_features))
    self.bias = nn.Parameter(torch.zeros(out_features))


    nn.init.kaiming_uniform_(self.weights, nonlinearity = 'relu')

  def forward(self, x):

    linear_out = torch.matmul(x, self.weights.t()) + self.bias
    return torch.relu(linear_out)

low_model = DenseLowLevel(in_features=10, out_features=5)
low_model

DenseLowLevel()

In [7]:
import torch
from torch.utils.data import Dataset
import numpy as np

class CoverTypeDataset(Dataset):

  def __init__(self, features, targets):

    self.x = torch.tensor(features, dtype = torch.float32)
    self.y = torch.tensor(targets, dtype = torch.long) - 1 # The pytorch expects the target to start from 0

  def __len__(self):
    return len(self.y)


  def __getitem__(self, idx):
    return self.x[idx], self.y[idx]


In [8]:
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

X_raw, y_raw = fetch_covtype(return_X_y = True)


print(f"Original Data Shape: {X_raw.shape}")
print(f"Original Targets: {np.unique(y_raw)}")

X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size = 0.2, random_state = 42, stratify = y_raw)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

train_dataset = CoverTypeDataset(X_train_scaled, y_train)
test_dataset = CoverTypeDataset(X_test_scaled, y_test)

sample_x, sample_y = train_dataset[0]

Original Data Shape: (581012, 54)
Original Targets: [1 2 3 4 5 6 7]


In [9]:
from torch.utils.data import DataLoader, random_split
import os

total_count = len(train_dataset)
val_count = int(0.1 * total_count)
train_count = total_count - val_count

train_subset, val_subset = random_split(train_dataset, [train_count, val_count],
                                          generator = torch.Generator().manual_seed(42))

BATCH_SIZE = 1024
NUM_WORKERS = min(os.cpu_count(), 4)
PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    train_subset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = NUM_WORKERS,
    pin_memory = PIN_MEMORY,
    persistent_workers = True if NUM_WORKERS > 0 else False
)



In [10]:
val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True if NUM_WORKERS > 0 else False
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)


In [11]:
import torch
import torch.nn as nn

class CustomDense(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.relu = nn.ReLU()

        self.bn = nn.BatchNorm1d(out_features)

    def forward(self, x):
        x = self.linear(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

In [12]:
class CoverTypeClassifier(nn.Module):
    def __init__(self, input_dim=54, num_classes=7, hidden_dim=128):
        super().__init__()

        self.layer1 = CustomDense(input_dim, hidden_dim)
        self.dropout1 = nn.Dropout(p=0.3)

        self.layer2 = CustomDense(hidden_dim, hidden_dim // 2)
        self.dropout2 = nn.Dropout(p=0.3)

        self.output_layer = nn.Linear(hidden_dim // 2, num_classes)

    def forward(self, x):

        x = self.layer1(x)
        x = self.dropout1(x)

        x = self.layer2(x)
        x = self.dropout2(x)

        x = self.output_layer(x)

        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CoverTypeClassifier(
    input_dim=54,
    num_classes=7
)

model = model.to(device)

print(model)

CoverTypeClassifier(
  (layer1): CustomDense(
    (linear): Linear(in_features=54, out_features=128, bias=True)
    (relu): ReLU()
    (bn): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (dropout1): Dropout(p=0.3, inplace=False)
  (layer2): CustomDense(
    (linear): Linear(in_features=128, out_features=64, bias=True)
    (relu): ReLU()
    (bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (dropout2): Dropout(p=0.3, inplace=False)
  (output_layer): Linear(in_features=64, out_features=7, bias=True)
)


In [11]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 36.7 MB/s eta 0:00:00


In [13]:
import torch.optim as optim
import torch.nn as nn
import optuna


def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_x, batch_y in loader:

        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        logits = model(batch_x)
        loss = criterion(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy



In [14]:
def evaluate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            running_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()

    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

In [14]:

def objective(trial):

    hidden_dim = trial.suggest_int("hidden_dim", 64, 512)

    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    optimizer_name = trial.suggest_categorical("optimizer", ["AdamW", "RMSprop"])

    dropout_p = trial.suggest_float("dropout_p", 0.1, 0.5)


    model = CoverTypeClassifier(
        input_dim=54,
        num_classes=7,
        hidden_dim=hidden_dim
    ).to(device)


    model.dropout1.p = dropout_p
    model.dropout2.p = dropout_p

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "AdamW":
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    else:
        optimizer = optim.RMSprop(model.parameters(), lr=lr)


    for epoch in range(15):

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

        val_loss, val_acc = evaluate(model, val_loader, criterion, device)


        trial.report(val_acc, epoch)


        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return val_acc



pruner_config = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)

study = optuna.create_study(
    direction="maximize",
    pruner=pruner_config
)

study.optimize(objective, n_trials=20)


print(f"Best Trial Accuracy: {study.best_trial.value:.2f}%")
print("Best Parameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

[I 2026-01-08 07:07:09,096] A new study created in memory with name: no-name-31dd0b0b-90d0-46b4-a830-20fd972aebd6
[I 2026-01-08 07:08:34,539] Trial 0 finished with value: 85.96815834767642 and parameters: {'hidden_dim': 324, 'lr': 0.0009916035793817249, 'optimizer': 'AdamW', 'dropout_p': 0.3688914927499829}. Best is trial 0 with value: 85.96815834767642.
[I 2026-01-08 07:09:45,065] Trial 1 finished with value: 84.94191049913941 and parameters: {'hidden_dim': 511, 'lr': 0.00026828523507855194, 'optimizer': 'RMSprop', 'dropout_p': 0.2784393569169652}. Best is trial 0 with value: 85.96815834767642.
[I 2026-01-08 07:10:53,697] Trial 2 finished with value: 88.6617900172117 and parameters: {'hidden_dim': 322, 'lr': 0.0029633417283724817, 'optimizer': 'AdamW', 'dropout_p': 0.26577817103096285}. Best is trial 2 with value: 88.6617900172117.
[I 2026-01-08 07:12:03,032] Trial 3 finished with value: 89.48795180722891 and parameters: {'hidden_dim': 257, 'lr': 0.008261489240188028, 'optimizer': 'RM

Best Trial Accuracy: 90.34%
Best Parameters:
  hidden_dim: 434
  lr: 0.0068034017286456895
  optimizer: RMSprop
  dropout_p: 0.21557383456866672


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import time


BEST_PARAMS = {
    'hidden_dim': 434,
    'dropout_p': 0.21557383456866672,
    'lr': 0.0068034017286456895,
    'optimizer': 'RMSprop'
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

final_model = CoverTypeClassifier(
    input_dim=54,
    num_classes=7,
    hidden_dim=BEST_PARAMS['hidden_dim']
).to(device)

final_model.dropout1.p = BEST_PARAMS['dropout_p']
final_model.dropout2.p = BEST_PARAMS['dropout_p']

print(f"Model Created with Hidden Dim: {BEST_PARAMS['hidden_dim']}")

if BEST_PARAMS['optimizer'] == 'RMSprop':
    optimizer = optim.RMSprop(final_model.parameters(), lr=BEST_PARAMS['lr'])
else:
    optimizer = optim.AdamW(final_model.parameters(), lr=BEST_PARAMS['lr'])

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.1, patience=3
)

criterion = nn.CrossEntropyLoss()

EPOCHS = 50
best_acc = 0.0

print("\n--- Starting Final Training ---")
start_time = time.time()

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(final_model, train_loader, criterion, optimizer, device)

    val_loss, val_acc = evaluate(final_model, val_loader, criterion, device)


    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(final_model.state_dict(), "final_model.pth")

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | LR: {optimizer.param_groups[0]['lr']:.6f}")

print(f"\nTraining Finished. Best Validation Accuracy: {best_acc:.2f}%")

final_model.load_state_dict(torch.load("final_model_93_target.pth"))
test_loss, test_acc = evaluate(final_model, test_loader, criterion, device)
print(f"FINAL TEST ACCURACY: {test_acc:.2f}%")

Model Created with Hidden Dim: 434

--- Starting Final Training ---
Epoch 1/50 | Train Acc: 74.86% | Val Acc: 78.07% | LR: 0.006803
Epoch 2/50 | Train Acc: 79.99% | Val Acc: 82.23% | LR: 0.006803
Epoch 3/50 | Train Acc: 82.26% | Val Acc: 84.78% | LR: 0.006803
Epoch 4/50 | Train Acc: 83.59% | Val Acc: 86.06% | LR: 0.006803
Epoch 5/50 | Train Acc: 84.56% | Val Acc: 86.31% | LR: 0.006803
Epoch 6/50 | Train Acc: 85.23% | Val Acc: 87.75% | LR: 0.006803
Epoch 7/50 | Train Acc: 85.75% | Val Acc: 87.86% | LR: 0.006803
Epoch 8/50 | Train Acc: 86.20% | Val Acc: 88.05% | LR: 0.006803
Epoch 9/50 | Train Acc: 86.58% | Val Acc: 88.81% | LR: 0.006803
Epoch 10/50 | Train Acc: 86.86% | Val Acc: 89.50% | LR: 0.006803
Epoch 11/50 | Train Acc: 87.15% | Val Acc: 89.72% | LR: 0.006803
Epoch 12/50 | Train Acc: 87.42% | Val Acc: 89.90% | LR: 0.006803
Epoch 13/50 | Train Acc: 87.60% | Val Acc: 90.19% | LR: 0.006803
Epoch 14/50 | Train Acc: 87.77% | Val Acc: 90.04% | LR: 0.006803
Epoch 15/50 | Train Acc: 87.90%

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

full_train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_final = CoverTypeClassifier(
    input_dim=54,
    num_classes=7,
    hidden_dim=434
).to(device)

model_final.dropout1.p = 0.2155
model_final.dropout2.p = 0.2155


optimizer = optim.RMSprop(model_final.parameters(), lr=0.0068)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.1, patience=3
)
criterion = nn.CrossEntropyLoss()

EPOCHS = 50

print(f"--- Retraining on full dataset ({len(train_dataset)} samples) ---")

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model_final, full_train_loader, criterion, optimizer, device)


    test_loss, test_acc = evaluate(model_final, test_loader, criterion, device)

    scheduler.step(test_acc)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

print(f"FINAL RESULT: {test_acc:.2f}%")

--- Retraining on full dataset (464809 samples) ---
Epoch 1/50 | Train Acc: 75.21% | Test Acc: 80.69%
Epoch 2/50 | Train Acc: 80.51% | Test Acc: 83.96%
Epoch 3/50 | Train Acc: 82.87% | Test Acc: 85.91%
Epoch 4/50 | Train Acc: 84.24% | Test Acc: 86.67%
Epoch 5/50 | Train Acc: 85.07% | Test Acc: 87.54%
Epoch 6/50 | Train Acc: 85.79% | Test Acc: 88.46%
Epoch 7/50 | Train Acc: 86.31% | Test Acc: 88.83%
Epoch 8/50 | Train Acc: 86.79% | Test Acc: 89.34%
Epoch 9/50 | Train Acc: 87.04% | Test Acc: 89.72%
Epoch 10/50 | Train Acc: 87.29% | Test Acc: 89.85%
Epoch 11/50 | Train Acc: 87.52% | Test Acc: 90.09%
Epoch 12/50 | Train Acc: 87.75% | Test Acc: 90.49%
Epoch 13/50 | Train Acc: 87.96% | Test Acc: 90.56%
Epoch 14/50 | Train Acc: 88.11% | Test Acc: 90.51%
Epoch 15/50 | Train Acc: 88.24% | Test Acc: 90.76%
Epoch 16/50 | Train Acc: 88.42% | Test Acc: 90.89%
Epoch 17/50 | Train Acc: 88.53% | Test Acc: 90.79%
Epoch 18/50 | Train Acc: 88.63% | Test Acc: 91.18%
Epoch 19/50 | Train Acc: 88.73% | Test 